# Paso 3 de Garrido — shard `s3_r2r_a` (R2r, semillas 1422001+)

**MPC de replay y DDMRP proyectado contra las 216 posturas estáticas**, sobre el contrato expandido
de buffers (`op3_rm`, `op5_rm`, `op9_rations`).

Este shard corre **6 tapes x 5 escenarios** de la familia **R2r**,
horizonte 52 semanas, época 4. Los cuatro shards difieren **sólo en qué
tapes tocan**; el contraste pareado vive dentro del shard y el análisis agrupado concatena filas.

**GPU apagada a propósito:** el cuello es el DES en Python puro, y los kernels GPU de Kaggle dan
menos vCPU.

**Métrica decisora `ret_excel_full_ledger`, no el default del runner.** `ret_excel` está medido premiando el
abandono, así que un controlador podría ganar dejando de servir.

Preregistro: `docs/PREREGISTRO_PASO3_GARRIDO_MPC_EXPANDIDO_2026-08-06.md`


In [ ]:
# 1) Repo + dependencias
import os, subprocess, sys, time, json, shutil
from pathlib import Path

ROOT = Path('/kaggle/working/scres-ia')
if not ROOT.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'codex/expanded-contract-comparators-v2',
                           'https://github.com/Thom-320/scres-ia.git', str(ROOT)])
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'simpy>=4.1', 'numpy', 'pandas'])
print('commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())
print('cpus  :', os.cpu_count())

In [ ]:
# 2) El shard. Todo lo que decide algo va explícito en la línea de comandos.
WORKERS = max(1, (os.cpu_count() or 2) - 1)
cmd = [sys.executable, 'scripts/run_expanded_contract_comparators_v2.py',
       '--phase', 'full',
       '--families', 'R2r',
       '--tapes', '6',
       '--scenarios', '5',
       '--seed-start', '1422001',
       '--horizon-weeks', '52',
       '--epoch-weeks', '4',
       '--metric', 'ret_excel_full_ledger',
       '--workers', str(WORKERS),
       '--output-dir', 'results/step3_s3_r2r_a']
print(' '.join(cmd), flush=True)
t0 = time.time()
subprocess.check_call(cmd)
print(f'listo en {time.time() - t0:.0f}s')

In [ ]:
# 3) Lectura rápida y ZIP para enviar
res = json.loads(Path('results/step3_s3_r2r_a/result.json').read_text())
print('claim_status:', res.get('claim_status'))
print('metric      :', res.get('metric'))
for fam, block in res.get('family_results', {}).items():
    print(f'\n== {fam} · {block.get("candidate_count")} candidatos')
    for arm, c in block.get('comparisons', {}).items():
        ci = c.get('ci95', [None, None])
        print(f'   {arm:<26} delta {c.get("delta_mean")}  IC95 {ci}')

shutil.make_archive('/kaggle/working/step3_s3_r2r_a', 'zip', 'results/step3_s3_r2r_a')
print('\nZIP -> /kaggle/working/step3_s3_r2r_a.zip')